In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


# **AfriSenti Error Analysis on the Twi Language**
## 1. Setup



In [ ]:
# Imports
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## 2. Configuration and Data Loading


In [ ]:
# Configuration
import warnings
warnings.filterwarnings('ignore')

# Paths (UPDATE WITH YOUR DRIVE PATH)
DRIVE_BASE = "/content/drive/MyDrive/AfriSenti/data"
LANG = "twi"  # Test language: twi, hau, yor, etc.

print(f"Loading data for {LANG}...")

# Load from Google Drive (already downloaded)
dataset = load_from_disk(f"{DRIVE_BASE}/{LANG}")

# Display dataset sizes
print(f"Train: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test: {len(dataset['test'])}")

# Display a raw example
print("\nRaw data example:")
print(f"Text: {dataset['train'][0]['tweet']}")
print(f"Label: {dataset['train'][0]['label']}")

## 3. Baseline (Seed 2)



In [ ]:
# 1. Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Davlan/afro-xlmr-base")
model = AutoModelForSequenceClassification.from_pretrained("Davlan/afro-xlmr-base", num_labels=3)

# 2. Tokenize
def tok(x):
    return tokenizer(x["tweet"], padding="max_length", truncation=True, max_length=200)

dataset = dataset.map(tok, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 3. Train
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        "./out", num_train_epochs=10, per_device_train_batch_size=32,
        learning_rate=1e-5, seed=2, eval_strategy="epoch",
        save_strategy="epoch", load_best_model_at_end=True, report_to="none",
    ),
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=lambda x: {"accuracy": accuracy_score(np.argmax(x.predictions, 1), x.label_ids)},
)

trainer.train()

# 4. Detailed evaluation
preds = trainer.predict(dataset["test"])
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}")
print("\nPer-class metrics:")
print(classification_report(y_true, y_pred, target_names=["pos", "neu", "neg"]))

# 5. Save model
model.save_pretrained("/content/drive/MyDrive/AfriSenti/models/twi_baseline_seed2")
tokenizer.save_pretrained("/content/drive/MyDrive/AfriSenti/models/twi_baseline_seed2")
print("\n Model saved to Drive")

## 4. Error Analysis

In [ ]:
# Run the error analysis script
!python /content/drive/MyDrive/AfriSenti/error_analysis.py

## **5. Interpretation of Results**

### Overall Performance
The model achieves **64.6% accuracy** on the test set. This is a reasonable baseline for a low-resource African language (random baseline would be 33%).

### Critical Issue: Neutral Class is Invisible
The model finds almost no neutral tweets. On the validation set, neutral recall is 0%. On the test set, it is only 1%. The neutral class is practically ignored.

**Why?** The neutral class is underrepresented (only 15% of the test set) and its examples are often ambiguous.

### Error Patterns
Among 135 validation errors:

- **Negative misclassified as positive**: 44 times
- **Neutral misclassified as positive**: 39 times
- **Positive misclassified as negative**: 33 times
- **Neutral misclassified as negative**: 19 times

**Key insight**: Neutral tweets are absorbed by both positive and negative classes. The model never keeps them as "neutral".

### What Causes the Errors?
- **Emojis** appear in 43% of errors. Smiling emojis (😂, 😁) dominate the model's prediction, pushing it toward "positive" even when the text is negative.
- **Repeated characters** appear in 31% of errors. Emphasis markers like "ooooo" and "paaa" are ignored by the model.
- **Exclamation and question marks** cause 0% of errors. These patterns work well and should be kept.

### Example of a Typical Error
> *"5mins biaa na scam nam mu 😁"* (True: negative → Predicted: positive)

The text contains "scam" (negative), but the 😁 emoji overrides the model's prediction.

### Conclusion
The model's main limitation is **data-related, not architectural**. The neutral class is underrepresented and emojis often dominate text sentiment.